# Kalan GPU deneyleri — tek çalıştırma

**1)** **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**

**2)** **Çalışma zamanı → Tümünü çalıştır** (uyarı çıkarsa *Yine de çalıştır*), Drive iznini ver.

Sırayla yapılanlar (her adım sonucunu Drive'a yazar; oturum koparsa **Tümünü çalıştır** yeter, biten kısımlar atlanır):

| Adım | İş | Süre (T4) |
|---|---|---|
| 4 | Duman testi — hata varsa burada durur | < 1 dk |
| 5 | ECG-FM donmuş (sonda) | ~5 dk |
| 6 | ECG-FM ince ayar: önceden eğitilmiş + rastgele, 12 bitki | ~1,5–2 sa |
| 7 | HuBERT-ECG ince ayar, 30 dk pencere (yarım kalan) | ~50 dk |
| 8 | Özet | < 1 dk |

Ayarlar deney öncesi sabittir (`src/bitki_ekg/ince_ayar.py`); değiştirme.

## 1. GPU kontrolü

In [ ]:
import torch
assert torch.cuda.is_available(), (
    'GPU yok! Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU seç, sonra Tümünü çalıştır.')
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
KOK = Path('/content/drive/MyDrive/bitkiEkg_sonuclar')
KOK.mkdir(parents=True, exist_ok=True)
print('Sonuç klasörü:', KOK)

## 3. Kod, kütüphaneler ve veri

In [ ]:
!rm -rf /content/bitkiEkg /content/fairseq-signals
!git clone -q -b onDeneme --depth 1 https://github.com/melihakcam/bitkiEkg.git /content/bitkiEkg
!git clone -q https://github.com/Jwoo5/fairseq-signals.git /content/fairseq-signals
!cd /content/fairseq-signals && git checkout -q f8f0ff1c788a82c2059cb452cd5462898867489e
!pip install -q cffi hydra-core omegaconf regex "sacrebleu>=1.4.12" wfdb "transformers>=4.40,<5" huggingface_hub
import sys
sys.path.insert(0, '/content/bitkiEkg/src')
sys.path.insert(0, '/content/fairseq-signals')
import transformers
assert int(transformers.__version__.split('.')[0]) < 5, 'transformers 5 yüklü: Çalışma zamanı → Yeniden başlat, sonra Tümünü çalıştır.'
from fairseq_signals.models import build_model_from_checkpoint

import numpy as np
from bitki_ekg.ekg_girdi import ecgfm_girdisi, hubert_girdisi
from bitki_ekg.ince_ayar import AYARLAR, Siniflandirici, ecgfm_yukle, lopo_calistir
from bitki_ekg.sonda import ecgfm_gomme, sonda_degerlendir

def veri(pencere):
    z = np.load(f'/content/bitkiEkg/data/colab/domates_ikili_{pencere}.npz')
    return z['Z'].astype(np.float32), z['y'].astype(int), z['bitki'].astype(int)

Z1, y1, g1 = veri('1h')
X_ecgfm = ecgfm_girdisi(Z1)
print('Hazır | ECG-FM girdi:', X_ecgfm.shape, '| sınıflar:', np.bincount(y1), '| bitki:', sorted(set(g1)))
print('Ayarlar:', AYARLAR)

## 4. Duman testi

In [ ]:
for tur, X in (('ecgfm', X_ecgfm[:4]), ('hubert', hubert_girdisi(Z1[:4]))):
    for onceden in (True, False):
        m = Siniflandirici(onceden, AYARLAR, tur).cuda().eval()
        with torch.no_grad():
            c = m(torch.from_numpy(X).cuda())
        assert c.shape == (4, 2) and torch.isfinite(c).all(), f'{tur} çıktısı hatalı'
        print(f'{tur:6s} {"önceden eğitilmiş" if onceden else "rastgele":18s} OK')
        del m; torch.cuda.empty_cache()
print('Duman testi geçti.')

## 5. ECG-FM donmuş (sonda)

In [ ]:
import pandas as pd
SONDA = KOK / 'ecgfm' / 'sonda_1h.csv'
if SONDA.exists():
    print('Önceden tamamlanmış, atlandı:', SONDA)
else:
    parcalar = []
    for ad, onceden in (('onceden_egitilmis', True), ('rastgele_baslatilmis', False)):
        E = ecgfm_gomme(ecgfm_yukle(onceden, AYARLAR.tohum), X_ecgfm, 'cuda')
        parcalar.append(sonda_degerlendir(E, y1, g1).assign(model=ad))
        print(ad, 'gömme', E.shape)
        torch.cuda.empty_cache()
    SONDA.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(parcalar).to_csv(SONDA, index=False)
s = pd.read_csv(SONDA)
print((s.groupby(['model', 'bolme'])[['dogruluk', 'auc']].mean() * 100).round(1))

## 6. ECG-FM ince ayar (1 saatlik pencere)

In [ ]:
DENEYLER = [('onceden_egitilmis', True), ('rastgele_baslatilmis', False)]
for ad, onceden in DENEYLER:
    print(f'=== ECG-FM {ad} ===', flush=True)
    lopo_calistir(X_ecgfm, y1, g1, onceden, KOK / 'ecgfm' / '1h' / ad, 'cuda', model_turu='ecgfm')
del X_ecgfm; torch.cuda.empty_cache()

## 7. HuBERT-ECG ince ayar (30 dakikalık pencere)

Önceki oturumda yarım kalan kısım; biten bitkiler atlanır.

In [ ]:
Z30, y30, g30 = veri('30min')
X30 = hubert_girdisi(Z30)
for ad, onceden in DENEYLER:
    print(f'=== HuBERT 30 dk {ad} ===', flush=True)
    lopo_calistir(X30, y30, g30, onceden, KOK / '30min' / ad, 'cuda')

## 8. Özet (bitince bana "bitti" yazman yeterli)

In [ ]:
import json
def ozet(klasor, ad_dosya):
    satir = []
    for ad, _ in DENEYLER:
        for f in sorted((klasor / ad).glob('bitki_*.json')):
            r = json.loads(f.read_text(encoding='utf-8'))
            satir.append({'deney': ad, 'bitki': r['test_bitkisi'], 'dogruluk': r['dogruluk'],
                          'f1': r['f1'], 'auc': r['auc'], 'en_iyi_epoch': r['en_iyi_epoch']})
    df = pd.DataFrame(satir)
    df.to_csv(KOK / ad_dosya, index=False)
    print(ad_dosya, '| tamamlanan:', df.groupby('deney').size().to_dict())
    print((df.groupby('deney')[['dogruluk', 'f1', 'auc']].agg(['mean', 'std']) * 100).round(1), '\n')

ozet(KOK / 'ecgfm' / '1h', 'ozet_ecgfm_1h.csv')
ozet(KOK / '30min', 'ozet_30min.csv')
print('Karşılaştırma (LOPO doğruluk, 1 sa): LightGBM 72,9 · MiniRocket 72,5 · HuBERT ince ayar 71,1')